<a href="https://colab.research.google.com/github/mohammadreza-mohammadi94/Agentic-AI-LLM-Apps/blob/main/Audio_Processing_HF/15_STST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part I: Speech Translation

In [1]:
import torch
from transformers import pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"

pipe = pipeline(
    "automatic-speech-recognition",
    model = "openai/whisper-base",
    device=device
)

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  290MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

... (3 outputs left out)

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

## `es` Dataset

In [3]:
from datasets import load_dataset

dataset = load_dataset("qmeeus/voxpopuli", "es", split="validation", streaming=True)
sample = next(iter(dataset))

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/74 [00:00<?, ?it/s]

In [4]:
sample

{'audio_id': '20090422-0900-PLENARY-18-es_20090422-21:34:44_0',
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x7cc090d26510>,
 'text': 'Señor Presidente, a diferencia de las dos oradoras que me acaban de preceder, yo estoy de acuerdo con la mayor parte de los intervinientes.',
 'language': 'es'}

In [6]:
# برای اینکه ویسپر به جای تشخیص گفتار (Transcription)، عمل ترجمه (Translation) را انجام دهد،
def translate(audio):
    outputs = pipe(audio, max_new_tokens=256, generate_kwargs={"task": "translate"})
    return outputs["text"]


# translated_text = translate(sample["audio"].copy())
# print(translated_text)


audio_input = {
    "raw": sample["audio"]["array"],
    "sampling_rate": sample["audio"]["sampling_rate"],
}

translated_text = translate(audio_input)
print(translated_text)

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 Mr. President, the differences of the two hours I have been given to Mr. President, I agree with most of the interventions.


# Part II: Text-To-Speech

In [7]:
from transformers import (
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan,
    SpeechT5Processor,
)

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

model.to(device)
vocoder.to(device)

preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model: reconstructing file:   0%|          |  0.00B /  238kB            

spm_char.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

... (9 outputs left out)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 50.7MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

SpeechT5HifiGan(
  (conv_pre): Conv1d(80, 512, kernel_size=(7,), stride=(1,), padding=(3,))
  (upsampler): ModuleList(
    (0): ConvTranspose1d(512, 256, kernel_size=(8,), stride=(4,), padding=(2,))
    (1): ConvTranspose1d(256, 128, kernel_size=(8,), stride=(4,), padding=(2,))
    (2): ConvTranspose1d(128, 64, kernel_size=(8,), stride=(4,), padding=(2,))
    (3): ConvTranspose1d(64, 32, kernel_size=(8,), stride=(4,), padding=(2,))
  )
  (resblocks): ModuleList(
    (0): HifiGanResidualBlock(
      (convs1): ModuleList(
        (0): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(3,), dilation=(3,))
        (2): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(5,), dilation=(5,))
      )
      (convs2): ModuleList(
        (0-2): 3 x Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(1,))
      )
    )
    (1): HifiGanResidualBlock(
      (convs1): ModuleList(
        (0): Conv1d(256, 256,

In [9]:
# Speaker Embeddings

import torch
from datasets import load_dataset

# بارگذاری مستقیم نسخه Parquet بدون اجرای اسکریپت قدیمی
embeddings_dataset = load_dataset(
    "Matthijs/cmu-arctic-xvectors",
    split="validation",
    revision="refs/convert/parquet",
)

speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
print(speaker_embeddings.shape)  # خروجی: torch.Size([1, 512])

default/validation/0000.parquet: reconstructing file:   0%|          |  0.00B / 21.3MB            

default/validation/0000.parquet: downloading bytes:           |  0.00B            

Generating validation split: 0 examples [00:00, ? examples/s]

torch.Size([1, 512])


In [10]:
def synthesise(text):
  inputs = processor(text=text, return_tensors="pt")
  speech = model.generate_speech(
      inputs["input_ids"].to(device),
      speaker_embeddings.to(device),
      vocoder=vocoder,
  )
  return speech.cpu()

# Creating a STST Demo

In [11]:
import numpy as np

target_dtype = np.int16
max_range = np.iinfo(target_dtype).max


def speech_to_speech_translation(audio):
  translated_text = translate(audio)
  synthesised_speech = synthesise(translated_text)
  synthesised_speech = (synthesised_speech.numpy() * max_range).astype(
      np.int16
  )
  return 16000, synthesised_speech

In [ ]:
import gradio as gr

demo = gr.Blocks()

mic_translate = gr.Interface(
    fn=speech_to_speech_translation,
    inputs=gr.Audio(sources="microphone", type="filepath"),
    outputs=gr.Audio(label="Generated Speech", type="numpy"),
)

file_translate = gr.Interface(
    fn=speech_to_speech_translation,
    inputs=gr.Audio(sources="upload", type="filepath"),
    outputs=gr.Audio(label="Generated Speech", type="numpy"),
)

with demo:
  gr.TabbedInterface(
      [mic_translate, file_translate], ["Microphone", "Audio File"]
  )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c36fe955ede36d4424.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
